# DGA Detection — MiniLM vs DistilBERT: ONNX Export & Benchmark

**Goal:** Export two small language models to ONNX, quantize them to INT8,
and benchmark their inference speed so you can decide which one to deploy
on the Bluefield 3 SmartNIC (ARM A78 core).

**Models compared:**
| Model | Params | Hidden size | Layers | Best for |
|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 22M | 384 | 6 | Fast, small footprint |
| `distilbert-base-uncased` | 66M | 768 | 6 | Higher accuracy baseline |

**Pipeline per model:**
```
PyTorch weights → ONNX FP32 → ONNX INT8 (quantized)
                                    ↓
                          Deploy via ORT C API on ARM
```

> **Run each cell top to bottom.** Every cell has an explanation above it.
> You do not need to understand PyTorch internals — the comments explain
> every step in plain English.

## Section 1 — Install Dependencies

Run this once. If you already ran it before, it's safe to run again — pip
will just say 'already satisfied'.

In [ ]:
# Install everything needed for this notebook
# onnxruntime-gpu replaces the CPU-only onnxruntime to enable CUDAExecutionProvider.
# If this step fails with a conflict, run first:
#   pip uninstall onnxruntime -y

import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'torch', 'onnx', 'onnxruntime-gpu',
    'onnxscript', 'optimum[onnxruntime]', 'numpy', 'tabulate'
], check=True)
print('✓ All packages ready')

## Section 2 — Imports & Folder Setup

We import all libraries and create the `models/` folder where all output
files (ONNX graphs, vocab, metadata) will be saved.

In [2]:
import os, json, time, statistics, warnings
import numpy as np
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

warnings.filterwarnings('ignore')  # suppress noisy export warnings

# ── output folders ────────────────────────────────────────────────────────────
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(exist_ok=True)
(MODELS_DIR / 'minilm'    / 'tokenizer').mkdir(parents=True, exist_ok=True)
(MODELS_DIR / 'distilbert'/ 'tokenizer').mkdir(parents=True, exist_ok=True)

print('✓ Folders created:')
for p in sorted(MODELS_DIR.rglob('*')):
    print(f'  {p}')

c:\Users\Sergio\miniconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Folders created:
  models\distilbert
  models\distilbert\tokenizer
  models\minilm
  models\minilm\tokenizer
  models\tokenizer
  models\tokenizer\tokenizer.json
  models\tokenizer\tokenizer_config.json
  models\vocab.txt


In [ ]:
# ── GPU / CUDA availability check ─────────────────────────────────────────────
cuda_available = torch.cuda.is_available()

if cuda_available:
    gpu_name   = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}  ({gpu_mem_gb:.1f} GB VRAM)')
    print(f'CUDA: {torch.version.cuda}')
    GPU_PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
else:
    print('No CUDA GPU detected — GPU sessions will be skipped')
    GPU_PROVIDERS = ['CPUExecutionProvider']

CPU_PROVIDERS = ['CPUExecutionProvider']

# Confirm onnxruntime-gpu is installed (not the CPU-only build)
_avail = ort.get_available_providers()
print(f'ORT providers: {_avail}')
if cuda_available and 'CUDAExecutionProvider' not in _avail:
    print('WARNING: CUDAExecutionProvider missing.')
    print('  Run: pip uninstall onnxruntime -y && pip install onnxruntime-gpu')
    cuda_available = False
    GPU_PROVIDERS  = CPU_PROVIDERS

## Section 3 — Test Domains

We use a fixed set of domain names for all experiments. These are **not
malicious** — we are only testing inference speed and output consistency,
not detection accuracy. The domains cover:

- **Short legit domains** (`google.com`) — minimal tokens, fastest inference
- **Long subdomains** (`edge-star-shv-01-iad3.facebook.com`) — stress test
- **Hyphenated / CDN patterns** — common in real DNS traffic
- **Pseudo-random looking** (`xkqpzmvtlrjbd.ru`) — simulates DGA output
- **Dictionary-based DGA style** (`translateincoming.com`) — harder to detect

The three domains in `EXPORT_DOMAINS` are used during ONNX export as a
calibration shape. All 20 in `BENCH_DOMAINS` are used for benchmarking.

In [3]:
# Used during ONNX export to show the model what input shapes look like
EXPORT_DOMAINS = [
    'google.com',
    'xkqpzmvtlrjbd.ru',          # pseudo-random (DGA style)
    'translateincoming.com',      # dictionary-based DGA style
]

# Full benchmark set — mix of lengths and patterns
BENCH_DOMAINS = [
    # short / common
    'google.com', 'github.com', 'amazon.com', 'reddit.com',
    'cloudflare.com', 'fastly.net', 'wikipedia.org',
    # medium — subdomain patterns
    'mail.google.com', 'api.github.com', 'security.ubuntu.com',
    'fonts.googleapis.com', 'cdn.jsdelivr.net',
    # long — stress test for token budget
    'edge-star-shv-01-iad3.facebook.com',
    'login.microsoftonline.com',
    'ec2-54-234-12-1.compute.amazonaws.com',
    'static.cloudflareinsights.com',
    # DGA-style
    'xkqpzmvtlrjbd.ru',
    'translateincoming.com',
    'mortiscontrastatim.com',
    'very-long-subdomain-for-testing-purposes.example.com',
]

print(f'Export domains : {len(EXPORT_DOMAINS)}')
print(f'Benchmark domains: {len(BENCH_DOMAINS)}')

Export domains : 3
Benchmark domains: 20


## Section 4 — Helper Functions

Three small utilities used throughout the notebook:

- **`mean_pool`** — takes the transformer's output (one vector per token)
  and collapses it to a single vector per domain by averaging, ignoring
  padding tokens. This is the domain embedding.

- **`cosine_sim`** — measures how similar two vectors are. Returns 1.0 if
  they are identical, 0.0 if they are perpendicular. We use this to check
  that ONNX and PyTorch produce the same result.

- **`benchmark`** — runs a function N times and returns latency statistics
  (mean, median p50, p95, p99). The first `n_warmup` runs are discarded
  because the first call always includes JIT/graph-loading overhead.

In [4]:
def mean_pool(last_hidden_state: np.ndarray,
              attention_mask:    np.ndarray) -> np.ndarray:
    """
    Collapse [batch, seq_len, hidden] → [batch, hidden] by averaging
    over the real (non-padding) tokens only.
    """
    mask = attention_mask[..., None].astype(np.float32)   # [B, S, 1]
    summed = (last_hidden_state * mask).sum(axis=1)        # [B, H]
    counts = mask.sum(axis=1)                              # [B, 1]
    return summed / (counts + 1e-9)                        # [B, H]


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity between two 1-D embedding vectors.
    1.0 = identical direction, 0.0 = orthogonal, -1.0 = opposite.
    We expect > 0.999 between FP32 and INT8 to call quantization safe.
    """
    a = a.flatten().astype(np.float64)
    b = b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def benchmark(fn, n_warmup: int = 20, n_runs: int = 200) -> dict:
    """
    Time `fn()` for `n_runs` iterations after `n_warmup` warm-up calls.
    Returns a dict with mean, p50, p95, p99, min latencies in milliseconds.
    """
    for _ in range(n_warmup):       # warm up: let Python/ORT settle
        fn()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)  # convert to ms
    s = sorted(times)
    return {
        'mean': statistics.mean(times),
        'p50':  s[n_runs // 2],
        'p95':  s[int(0.95 * n_runs)],
        'p99':  s[int(0.99 * n_runs)],
        'min':  s[0],
    }

print('✓ Helper functions defined')

✓ Helper functions defined


## Section 5 — Load Models from HuggingFace

We load both models. The first run downloads weights from HuggingFace
(~85MB for MiniLM, ~265MB for DistilBERT). Subsequent runs use the local
cache so it is instant.

**`model.eval()`** switches off dropout and batch-norm training behaviour.
Always call this before export or inference — without it you get different
outputs on every forward pass.

**Key size difference:**
- MiniLM hidden size = **384** → each domain embedding is a 384-dim vector
- DistilBERT hidden size = **768** → twice as wide, twice the compute

In [5]:
# ── MiniLM-L6-v2 ─────────────────────────────────────────────────────────────
MINILM_ID = 'sentence-transformers/all-MiniLM-L6-v2'
print(f'Loading {MINILM_ID} ...')
minilm_tok   = AutoTokenizer.from_pretrained(MINILM_ID)
minilm_model = AutoModel.from_pretrained(MINILM_ID)
minilm_model.eval()

minilm_params = sum(p.numel() for p in minilm_model.parameters())
print(f'  Parameters : {minilm_params/1e6:.1f}M')
print(f'  Hidden size: {minilm_model.config.hidden_size}')
print(f'  Layers     : {minilm_model.config.num_hidden_layers}')

# Save tokenizer for later use by the C inference binary
minilm_tok.save_pretrained(str(MODELS_DIR / 'minilm' / 'tokenizer'))
print(f'  Tokenizer saved ✓')

Loading sentence-transformers/all-MiniLM-L6-v2 ...
  Parameters : 22.7M
  Hidden size: 384
  Layers     : 6
  Tokenizer saved ✓


In [6]:
# ── DistilBERT-base-uncased ───────────────────────────────────────────────────
DISTILBERT_ID = 'distilbert-base-uncased'
print(f'Loading {DISTILBERT_ID} ...')
dbert_tok   = AutoTokenizer.from_pretrained(DISTILBERT_ID)
dbert_model = AutoModel.from_pretrained(DISTILBERT_ID)
dbert_model.eval()

dbert_params = sum(p.numel() for p in dbert_model.parameters())
print(f'  Parameters : {dbert_params/1e6:.1f}M')
print(f'  Hidden size: {dbert_model.config.hidden_size}')
print(f'  Layers     : {dbert_model.config.num_hidden_layers}')

dbert_tok.save_pretrained(str(MODELS_DIR / 'distilbert' / 'tokenizer'))
print(f'  Tokenizer saved ✓')

print(f'\nSize ratio DistilBERT / MiniLM: {dbert_params/minilm_params:.1f}×')

Loading distilbert-base-uncased ...
  Parameters : 66.4M
  Hidden size: 768
  Layers     : 6
  Tokenizer saved ✓

Size ratio DistilBERT / MiniLM: 2.9×


## Section 6 — PyTorch Forward Pass (Ground Truth)

Before exporting to ONNX, we run both models in PyTorch and save the
embeddings. These become our **ground truth** — later we measure how
closely ONNX FP32 and ONNX INT8 match them.

**What `torch.no_grad()` does:** tells PyTorch not to record operations
for backpropagation. This saves memory and speeds up the forward pass.
Always use it during inference.

In [8]:
MAX_LEN = 64

def pytorch_embed(model, tokenizer, domains):
    enc = tokenizer(
        domains,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )

    # DistilBERT does NOT accept token_type_ids — check before passing
    model_inputs = {
        'input_ids':      enc['input_ids'],
        'attention_mask': enc['attention_mask'],
    }
    # Only add token_type_ids if the model supports it
    if 'token_type_ids' in enc:
        model_inputs['token_type_ids'] = enc['token_type_ids']

    with torch.no_grad():
        out = model(**model_inputs)

    mask = enc['attention_mask'].unsqueeze(-1).float()
    emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
    return emb.numpy(), enc


print('Running PyTorch forward pass ...')
minilm_pt_emb,  minilm_enc   = pytorch_embed(minilm_model, minilm_tok, EXPORT_DOMAINS)
distilbert_pt_emb, dbert_enc = pytorch_embed(dbert_model,  dbert_tok,  EXPORT_DOMAINS)

print(f'  MiniLM embeddings     shape: {minilm_pt_emb.shape}')
print(f'  DistilBERT embeddings shape: {distilbert_pt_emb.shape}')
print(f'  MiniLM norm     (google.com): {np.linalg.norm(minilm_pt_emb[0]):.4f}')
print(f'  DistilBERT norm (google.com): {np.linalg.norm(distilbert_pt_emb[0]):.4f}')

Running PyTorch forward pass ...
  MiniLM embeddings     shape: (3, 384)
  DistilBERT embeddings shape: (3, 768)
  MiniLM norm     (google.com): 6.5668
  DistilBERT norm (google.com): 9.0709


## Section 7 — Export to ONNX

**What is ONNX?** Open Neural Network Exchange — a standard file format
that describes a neural network as a graph of operations. Once exported,
the model can run on any hardware that has an ONNX Runtime (ORT) — including
ARM64 CPUs, GPUs, FPGAs, and SmartNICs — without Python or PyTorch.

**Key export parameters:**
- `dynamic_axes` — tells ONNX the batch size and sequence length can change
  at runtime. Without this, the model only accepts the exact shape used
  during export.
- `do_constant_folding` — pre-computes constant subgraphs at export time
  (e.g. layer norm weights never change, so multiply them out once).
- `opset_version=18` — the ONNX operator set version. 18 is current stable.

**Note:** New versions of PyTorch (2.4+) split large models into
`model.onnx` (graph) + `model.onnx.data` (weights). This is normal —
both files must stay together.

In [ ]:
def export_to_onnx(model, tokenizer, export_domains, out_path, model_name):
    """
    Export a HuggingFace model to ONNX FP32.
    Returns the file size in MB.
    """
    print(f'Exporting {model_name} to ONNX ...')

    enc = tokenizer(
        export_domains, padding=True, truncation=True,
        max_length=MAX_LEN, return_tensors='pt'
    )

    # Extract tensors from the tokenizer output
    input_ids      = enc['input_ids']
    attention_mask = enc['attention_mask']

    # Build inputs dict — DistilBERT has no token_type_ids
    model_inputs = {
        'input_ids':      input_ids,
        'attention_mask': attention_mask,
    }
    input_names = ['input_ids', 'attention_mask']

    if 'token_type_ids' in enc:
        model_inputs['token_type_ids'] = enc['token_type_ids']
        input_names.append('token_type_ids')

    # dynamic_axes: only include token_type_ids if the model uses it
    dynamic_axes = {
        'input_ids':         {0: 'batch', 1: 'seq_len'},
        'attention_mask':    {0: 'batch', 1: 'seq_len'},
        'last_hidden_state': {0: 'batch', 1: 'seq_len'},
    }
    if 'token_type_ids' in enc:
        dynamic_axes['token_type_ids'] = {0: 'batch', 1: 'seq_len'}

    torch.onnx.export(
        model,
        args=tuple(model_inputs.values()),
        f=str(out_path),
        input_names=input_names,
        output_names=['last_hidden_state'],
        dynamic_axes=dynamic_axes,
        opset_version=18,
        do_constant_folding=True,
        verbose=False,
    )

    # Size calculation: if .onnx.data exists, count both files
    data_file = Path(str(out_path) + '.data')
    total_bytes = out_path.stat().st_size
    if data_file.exists():
        total_bytes += data_file.stat().st_size
    size_mb = total_bytes / 1e6
    print(f'  Saved → {out_path}  ({size_mb:.1f} MB total)')
    return size_mb


# ── Export both models ────────────────────────────────────────────────────────
MINILM_ONNX_FP32 = MODELS_DIR / 'minilm'      / 'minilm_fp32.onnx'
DBERT_ONNX_FP32  = MODELS_DIR / 'distilbert'  / 'distilbert_fp32.onnx'

minilm_fp32_mb = export_to_onnx(
    minilm_model, minilm_tok, EXPORT_DOMAINS,
    MINILM_ONNX_FP32, 'MiniLM-L6'
)

dbert_fp32_mb = export_to_onnx(
    dbert_model, dbert_tok, EXPORT_DOMAINS,
    DBERT_ONNX_FP32, 'DistilBERT'
)

print(f'\nSize comparison:')
print(f'  MiniLM FP32    : {minilm_fp32_mb:.1f} MB')
print(f'  DistilBERT FP32: {dbert_fp32_mb:.1f} MB')
print(f'  Ratio          : {dbert_fp32_mb/minilm_fp32_mb:.1f}×')

Exporting MiniLM-L6 to ONNX ...


NameError: name 'input_ids' is not defined

## Section 8 — INT8 Quantization

**What is quantization?** Normally model weights are stored as 32-bit floats
(FP32). Quantization converts them to 8-bit integers (INT8). This gives:
- **~4× smaller file** (8 bits vs 32 bits per weight)
- **~2–4× faster inference** on ARM (NEON INT8 GEMM is much faster)
- **~0.1–0.3% accuracy loss** — usually negligible for embeddings

**Dynamic quantization** (what we use here):
- Weights are quantized once at export time
- Activations (intermediate values) are quantized on-the-fly at runtime
- No calibration dataset needed

**`per_channel=True`** — each weight column gets its own scale factor
instead of one global scale. More accurate but slightly larger file.
This is critical for cosine similarity > 0.999.

**`op_types_to_quantize=['MatMul']`** — we only quantize the heavy matrix
multiplications. The final projection layer (`Gemm`) stays in FP32 to
preserve output accuracy.

In [ ]:
def quantize_to_int8(fp32_path, int8_path, model_name):
    """
    Apply dynamic INT8 quantization to an ONNX FP32 model.
    Returns the INT8 file size in MB.
    """
    print(f'Quantizing {model_name} to INT8 ...')

    quantize_dynamic(
        model_input  = str(fp32_path),
        model_output = str(int8_path),
        weight_type  = QuantType.QInt8,
        op_types_to_quantize = ['MatMul'],   # only quantize MatMul, keep Gemm FP32
        per_channel  = True,                 # per-channel scale → more accurate
        reduce_range = False,                # False = standard INT8 range for ARM
    )

    size_mb = int8_path.stat().st_size / 1e6
    print(f'  Saved → {int8_path}  ({size_mb:.1f} MB)')
    return size_mb


MINILM_ONNX_INT8 = MODELS_DIR / 'minilm'     / 'minilm_int8.onnx'
DBERT_ONNX_INT8  = MODELS_DIR / 'distilbert'  / 'distilbert_int8.onnx'

minilm_int8_mb = quantize_to_int8(MINILM_ONNX_FP32,  MINILM_ONNX_INT8,  'MiniLM-L6')
dbert_int8_mb  = quantize_to_int8(DBERT_ONNX_FP32,   DBERT_ONNX_INT8,   'DistilBERT')

print(f'\nCompression results:')
print(f'  MiniLM    FP32→INT8: {minilm_fp32_mb:.1f}MB → {minilm_int8_mb:.1f}MB  '
      f'({100*(1-minilm_int8_mb/minilm_fp32_mb):.0f}% smaller)')
print(f'  DistilBERT FP32→INT8: {dbert_fp32_mb:.1f}MB → {dbert_int8_mb:.1f}MB  '
      f'({100*(1-dbert_int8_mb/dbert_fp32_mb):.0f}% smaller)')

In [ ]:
# Session options — single-threaded to match DPDK lcore behaviour
def make_session(onnx_path, providers=None):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1
    opts.inter_op_num_threads = 1
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(
        str(onnx_path),
        sess_options=opts,
        providers=providers or CPU_PROVIDERS,
    )

# ── CPU sessions (x86, single-threaded, matches DPDK lcore) ──────────────────
print('Loading CPU sessions ...')
sess_minilm_fp32 = make_session(MINILM_ONNX_FP32, CPU_PROVIDERS)
sess_minilm_int8 = make_session(MINILM_ONNX_INT8, CPU_PROVIDERS)
sess_dbert_fp32  = make_session(DBERT_ONNX_FP32,  CPU_PROVIDERS)
sess_dbert_int8  = make_session(DBERT_ONNX_INT8,  CPU_PROVIDERS)
print('  4 CPU sessions ready ✓')

# ── GPU sessions (RTX 5070, CUDA EP) ─────────────────────────────────────────
# Dynamic INT8 (QLinearMatMul) may partially fall back to CPU on CUDA EP for
# ops not yet GPU-accelerated; the session still runs but check ORT warnings.
if cuda_available:
    print(f'Loading GPU sessions ({gpu_name}) ...')
    sess_minilm_fp32_gpu = make_session(MINILM_ONNX_FP32, GPU_PROVIDERS)
    sess_minilm_int8_gpu = make_session(MINILM_ONNX_INT8, GPU_PROVIDERS)
    sess_dbert_fp32_gpu  = make_session(DBERT_ONNX_FP32,  GPU_PROVIDERS)
    sess_dbert_int8_gpu  = make_session(DBERT_ONNX_INT8,  GPU_PROVIDERS)
    print('  4 GPU sessions ready ✓')
else:
    print('  GPU sessions skipped (CUDA unavailable)')


def onnx_embed(session, tokenizer, domains):
    """Run domains through an ONNX session → [N, hidden_size] embeddings."""
    enc = tokenizer(
        domains, padding=True, truncation=True,
        max_length=MAX_LEN, return_tensors='pt'
    )
    feeds = {
        'input_ids':      enc['input_ids'].numpy().astype(np.int64),
        'attention_mask': enc['attention_mask'].numpy().astype(np.int64),
    }
    if 'token_type_ids' in enc:
        feeds['token_type_ids'] = enc['token_type_ids'].numpy().astype(np.int64)
    hidden = session.run(['last_hidden_state'], feeds)[0]  # [N, S, H]
    return mean_pool(hidden, feeds['attention_mask'])       # [N, H]

print('✓ onnx_embed() ready')

## Section 10 — Numerical Equivalence Check

We compare the embeddings from PyTorch, ONNX FP32, and ONNX INT8 using
cosine similarity. The interpretation:

| Cosine value | Meaning |
|---|---|
| 1.000000 | Bit-identical outputs |
| > 0.9999 | Safe for deployment — drift is negligible |
| 0.999–0.9999 | Acceptable — tiny float rounding |
| < 0.999 | Problem — check quantization settings |

**PT↔FP32** should be ~1.0 (ONNX FP32 must match PyTorch exactly).  
**FP32↔INT8** should be >0.999 (quantization must not distort embeddings).

## Section 11 — Latency Benchmark: x86 CPU vs RTX 5070 (batch=1)

Single-domain inference — the DPDK lcore scenario: one domain in, one embedding out.

**Hardware under test:**
- **x86 CPU** — OnnxRuntime, 1 thread (`CPUExecutionProvider`)
- **RTX 5070** — OnnxRuntime CUDA EP (`CUDAExecutionProvider`)

**Important caveat for batch=1:** at single-domain size the GPU is often *slower* than
the CPU because:
1. Input tensors (~64 tokens) are tiny — PCIe DMA + kernel launch overhead dominates
2. CUDA kernel launch latency ≈ 10–50 µs, which is comparable to the CPU compute time
3. The GPU's parallelism only pays off when the batch is large enough to fill its SMs

The GPU advantage will appear in Section 12b (batch sweep).

**Q8 on GPU note:** dynamic INT8 quantization uses `QLinearMatMul` operators. CUDA EP
supports these since ORT 1.16+, but some ops may fall back to CPU. ORT prints a
warning per fallback op — expect a few lines of `EP fallback` on first session load.

In [ ]:
N_WARMUP = 20
N_RUNS   = 200
TEST_D   = ['google.com']   # batch=1

print(f'Benchmarking (N={N_RUNS} runs after warm-up) on single domain ...\n')

bench_results = {}

# ── Config list: (label, pytorch_model, tokenizer, ort_session, mode) ─────────
configs = [
    # x86 CPU ─────────────────────────────────────────────────────────────────
    ('MiniLM     PyTorch FP32 [CPU]', minilm_model, minilm_tok, None,                'pytorch'),
    ('MiniLM     ONNX    FP32 [CPU]', None,         minilm_tok, sess_minilm_fp32,    'onnx'),
    ('MiniLM     ONNX    Q8  [CPU]',  None,         minilm_tok, sess_minilm_int8,    'onnx'),
    ('DistilBERT PyTorch FP32 [CPU]', dbert_model,  dbert_tok,  None,                'pytorch'),
    ('DistilBERT ONNX    FP32 [CPU]', None,         dbert_tok,  sess_dbert_fp32,     'onnx'),
    ('DistilBERT ONNX    Q8  [CPU]',  None,         dbert_tok,  sess_dbert_int8,     'onnx'),
]

if cuda_available:
    configs += [
        # RTX 5070 (CUDA EP) ──────────────────────────────────────────────────
        ('MiniLM     ONNX    FP32 [GPU]', None,     minilm_tok, sess_minilm_fp32_gpu, 'onnx'),
        ('MiniLM     ONNX    Q8  [GPU]',  None,     minilm_tok, sess_minilm_int8_gpu, 'onnx'),
        ('DistilBERT ONNX    FP32 [GPU]', None,     dbert_tok,  sess_dbert_fp32_gpu,  'onnx'),
        ('DistilBERT ONNX    Q8  [GPU]',  None,     dbert_tok,  sess_dbert_int8_gpu,  'onnx'),
    ]

for label, pt_model, tok, sess, mode in configs:
    is_gpu = '[GPU]' in label
    n_wu   = 50 if is_gpu else N_WARMUP   # GPU needs more warm-up for CUDA kernel JIT

    if mode == 'pytorch':
        def fn(m=pt_model, t=tok):
            enc = t(TEST_D, padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors='pt')
            with torch.no_grad():
                m(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
    else:
        def fn(s=sess, t=tok):
            onnx_embed(s, t, TEST_D)

    stats = benchmark(fn, n_wu, N_RUNS)
    bench_results[label] = stats

    prefix = '⚡' if is_gpu else '  '
    print(f'{prefix} {label:<42}  p50={stats["p50"]:6.2f}ms  '
          f'p95={stats["p95"]:6.2f}ms  p99={stats["p99"]:6.2f}ms')

print('\n✓ Benchmark done')

In [ ]:
# Configuration
N_WARMUP = 20
N_RUNS   = 200
TEST_D   = ['google.com']   # single domain — batch=1

print(f'Benchmarking (N={N_RUNS} runs, {N_WARMUP} warm-up) on single domain ...')
print('This takes about 2 minutes.\n')

bench_results = {}

configs = [
    ('MiniLM    PyTorch FP32', minilm_model,    minilm_tok,  None,            'pytorch'),
    ('MiniLM    ONNX FP32',    None,             minilm_tok,  sess_minilm_fp32,'onnx'),
    ('MiniLM    ONNX INT8',    None,             minilm_tok,  sess_minilm_int8, 'onnx'),
    ('DistilBERT PyTorch FP32',dbert_model,      dbert_tok,   None,            'pytorch'),
    ('DistilBERT ONNX FP32',   None,             dbert_tok,   sess_dbert_fp32, 'onnx'),
    ('DistilBERT ONNX INT8',   None,             dbert_tok,   sess_dbert_int8,  'onnx'),
]

for label, pt_model, tok, sess, mode in configs:
    if mode == 'pytorch':
        def fn():
            enc = tok(TEST_D, padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors='pt')
            tt  = enc.get('token_type_ids', torch.zeros_like(enc['input_ids']))
            with torch.no_grad():
                pt_model(input_ids=enc['input_ids'],
                         attention_mask=enc['attention_mask'],
                         token_type_ids=tt)
    else:
        def fn(s=sess, t=tok):
            onnx_embed(s, t, TEST_D)

    stats = benchmark(fn, N_WARMUP, N_RUNS)
    bench_results[label] = stats
    print(f'  {label:<28}  p50={stats["p50"]:6.2f}ms  '
          f'p95={stats["p95"]:6.2f}ms  p99={stats["p99"]:6.2f}ms')

print('\n✓ Benchmark done')

## Section 12b — CPU vs GPU Batch Sweep (FP32 + Q8)

The GPU crossover point: below a certain batch size the CPU wins; above it the GPU
wins because it can saturate its CUDA cores.

We sweep both FP32 and Q8 models across batch sizes 1–128 for MiniLM and report
**per-domain latency** and **throughput** for each hardware path.

**Why the GPU wins at large batches:**  
The RTX 5070 has ~4600 CUDA cores. At batch=1 most of them sit idle while the CPU
finishes in a few ms. At batch=64+ the GPU attention matrix fills enough SMs to
outrun the CPU's serial execution.

In [ ]:
if not cuda_available:
    print('GPU not available — skipping CPU vs GPU batch sweep')
else:
    BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128]

    sweep_configs = [
        ('MiniLM  FP32 CPU', sess_minilm_fp32,     minilm_tok),
        ('MiniLM  FP32 GPU', sess_minilm_fp32_gpu,  minilm_tok),
        ('MiniLM  Q8  CPU',  sess_minilm_int8,     minilm_tok),
        ('MiniLM  Q8  GPU',  sess_minilm_int8_gpu,  minilm_tok),
    ]

    # ── per-batch-size, per-config: collect p50 per-domain latency ───────────
    sweep_data = {cfg[0]: {} for cfg in sweep_configs}

    for bs in BATCH_SIZES:
        batch = (BENCH_DOMAINS * 10)[:bs]
        for label, sess, tok in sweep_configs:
            is_gpu = 'GPU' in label
            def fn(s=sess, t=tok, b=batch):
                onnx_embed(s, t, b)
            st = benchmark(fn, n_warmup=30 if is_gpu else 10, n_runs=80)
            sweep_data[label][bs] = st['p50'] / bs          # ms per domain

    # ── print table ───────────────────────────────────────────────────────────
    col_labels = [cfg[0] for cfg in sweep_configs]
    header = f'  {"batch":>6}' + ''.join(f'  {c:>18}' for c in col_labels)
    print(header)
    print('  ' + '-' * (8 + 20 * len(col_labels)))
    for bs in BATCH_SIZES:
        row = f'  {bs:>6}'
        for label in col_labels:
            ms  = sweep_data[label][bs]
            dps = int(1000 / ms)
            row += f'  {ms:>7.3f}ms {dps:>6}d/s'
        print(row)

    # ── find crossover points ─────────────────────────────────────────────────
    print('\n  Crossover analysis (MiniLM):')
    for prec in ['FP32', 'Q8']:
        cpu_key = f'MiniLM  {prec} CPU'
        gpu_key = f'MiniLM  {prec} GPU'
        for bs in BATCH_SIZES:
            if sweep_data[gpu_key][bs] < sweep_data[cpu_key][bs]:
                print(f'  {prec}: GPU faster than CPU starting at batch={bs}')
                break
        else:
            print(f'  {prec}: CPU faster than GPU across all batch sizes tested')

In [ ]:
print('Batch size sweep (ONNX INT8 only) ...')
print(f'  {"batch":>6}  {"MiniLM ms/dom":>15}  {"MiniLM dom/s":>13}  '
      f'{"DistilBERT ms/dom":>18}  {"DistilBERT dom/s":>16}')
print('  ' + '-'*74)

batch_results = []
for bs in [1, 2, 4, 8, 16, 32]:
    batch = (BENCH_DOMAINS * 3)[:bs]

    def fn_m(b=batch): onnx_embed(sess_minilm_int8, minilm_tok, b)
    def fn_d(b=batch): onnx_embed(sess_dbert_int8,  dbert_tok,  b)

    sm = benchmark(fn_m, n_warmup=10, n_runs=100)
    sd = benchmark(fn_d, n_warmup=10, n_runs=100)

    m_per = sm['p50'] / bs;  m_dps = 1000 / m_per
    d_per = sd['p50'] / bs;  d_dps = 1000 / d_per

    batch_results.append({'batch': bs,
                          'minilm_per_domain_ms': m_per, 'minilm_dps': m_dps,
                          'dbert_per_domain_ms':  d_per, 'dbert_dps':  d_dps})

    print(f'  {bs:>6}  {m_per:>15.3f}  {m_dps:>13.0f}  {d_per:>18.3f}  {d_dps:>16.0f}')

# Find sweet spots
best_m = min(batch_results, key=lambda x: x['minilm_per_domain_ms'])
best_d = min(batch_results, key=lambda x: x['dbert_per_domain_ms'])
print(f'\n  Optimal batch size: MiniLM={best_m["batch"]}  DistilBERT={best_d["batch"]}')

In [ ]:
from tabulate import tabulate

# ── Main latency table ────────────────────────────────────────────────────────
rows = []
for label, stats in bench_results.items():
    model     = 'MiniLM-L6' if 'MiniLM' in label else 'DistilBERT'
    hw        = 'RTX 5070'  if '[GPU]'  in label else 'x86 CPU'
    backend   = 'PyTorch'   if 'PyTorch' in label else 'ONNX RT'
    precision = 'Q8'        if 'Q8'     in label else 'FP32'
    p50       = stats['p50']
    p99       = stats['p99']
    arm_est   = f'~{p50*0.6:.1f}ms' if (precision == 'Q8' and hw == 'x86 CPU') else '—'
    rows.append([model, hw, backend, precision, f'{p50:.2f}', f'{p99:.2f}', arm_est])

headers = ['Model', 'Hardware', 'Backend', 'Precision', 'p50 (ms)', 'p99 (ms)', 'ARM est.']
print(tabulate(rows, headers=headers, tablefmt='github'))

# ── CPU vs GPU speedup table (batch=1) ───────────────────────────────────────
if cuda_available:
    print('\n── CPU vs RTX 5070 speedup at batch=1 ──')
    pairs = [
        ('MiniLM  FP32',     'MiniLM     ONNX    FP32 [CPU]', 'MiniLM     ONNX    FP32 [GPU]'),
        ('MiniLM  Q8',       'MiniLM     ONNX    Q8  [CPU]',  'MiniLM     ONNX    Q8  [GPU]'),
        ('DistilBERT FP32',  'DistilBERT ONNX    FP32 [CPU]', 'DistilBERT ONNX    FP32 [GPU]'),
        ('DistilBERT Q8',    'DistilBERT ONNX    Q8  [CPU]',  'DistilBERT ONNX    Q8  [GPU]'),
    ]
    sp_rows = []
    for name, cpu_k, gpu_k in pairs:
        if cpu_k in bench_results and gpu_k in bench_results:
            cpu_p50 = bench_results[cpu_k]['p50']
            gpu_p50 = bench_results[gpu_k]['p50']
            ratio   = cpu_p50 / gpu_p50
            winner  = 'GPU' if ratio > 1.05 else ('CPU' if ratio < 0.95 else 'tie')
            sp_rows.append([name, f'{cpu_p50:.2f}ms', f'{gpu_p50:.2f}ms', f'{ratio:.2f}×', winner])
    print(tabulate(sp_rows,
                   headers=['Config', 'CPU p50', 'GPU p50', 'CPU/GPU ratio', 'Winner'],
                   tablefmt='github'))
    print('\n(ratio > 1 → GPU faster; ratio < 1 → CPU faster)')
    print('GPU advantage grows with batch size — see Section 12b.')

# ── File sizes ────────────────────────────────────────────────────────────────
print('\nModel file sizes:')
size_rows = [
    ['MiniLM-L6',  'FP32', f'{minilm_fp32_mb:.0f} MB'],
    ['MiniLM-L6',  'Q8',   f'{minilm_int8_mb:.0f} MB'],
    ['DistilBERT', 'FP32', f'{dbert_fp32_mb:.0f} MB'],
    ['DistilBERT', 'Q8',   f'{dbert_int8_mb:.0f} MB'],
]
print(tabulate(size_rows, headers=['Model', 'Format', 'Size'], tablefmt='github'))

In [ ]:
print('Per-domain latency (ONNX INT8, batch=1) ...')
print(f'  {"domain":<48} {"tokens":>6}  {"MiniLM ms":>10}  {"DistilBERT ms":>14}')
print('  ' + '-'*82)

per_domain = []
for d in BENCH_DOMAINS:
    tlen_m = len(minilm_tok([d])['input_ids'][0])
    tlen_d = len(dbert_tok([d])['input_ids'][0])

    sm = benchmark(lambda: onnx_embed(sess_minilm_int8, minilm_tok, [d]),
                   n_warmup=10, n_runs=80)
    sd = benchmark(lambda dd=d: onnx_embed(sess_dbert_int8,  dbert_tok,  [dd]),
                   n_warmup=10, n_runs=80)

    per_domain.append({'domain': d, 'tok_len': tlen_m,
                       'minilm_ms': sm['p50'], 'dbert_ms': sd['p50']})

    ratio = sd['p50'] / sm['p50']
    print(f'  {d:<48} {tlen_m:>6}  {sm["p50"]:>10.3f}  {sd["p50"]:>14.3f}  ({ratio:.1f}×)')

print('  ' + '-'*82)
avg_m = np.mean([r['minilm_ms'] for r in per_domain])
avg_d = np.mean([r['dbert_ms']  for r in per_domain])
print(f'  {"AVERAGE":<48} {"":>6}  {avg_m:>10.3f}  {avg_d:>14.3f}  ({avg_d/avg_m:.1f}×)')

In [ ]:
output = {
    'models': {
        'minilm':     {'id': MINILM_ID,    'params_M': minilm_params/1e6,
                       'hidden_size': 384, 'layers': 6},
        'distilbert': {'id': DISTILBERT_ID, 'params_M': dbert_params/1e6,
                       'hidden_size': 768, 'layers': 6},
    },
    'hardware': {
        'cpu': 'x86 (single-threaded ORT)',
        'gpu': gpu_name if cuda_available else None,
        'cuda_version': torch.version.cuda if cuda_available else None,
    },
    'file_sizes_mb': {
        'minilm_fp32':    minilm_fp32_mb,
        'minilm_int8':    minilm_int8_mb,
        'distilbert_fp32':dbert_fp32_mb,
        'distilbert_int8':dbert_int8_mb,
    },
    'equivalence': results,
    'latency_batch1_ms': {k: v for k, v in bench_results.items()},
    'batch_sweep_int8': batch_results,
    'batch_sweep_cpu_vs_gpu': sweep_data if cuda_available else {},
    'per_domain': per_domain,
}

out_path = MODELS_DIR / 'benchmark_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, default=str)

print(f'✓ Results saved → {out_path}')
print(f'  Keys: {list(output.keys())}')

In [ ]:
from tabulate import tabulate

# Build summary rows
rows = []
for label, stats in bench_results.items():
    model = 'MiniLM-L6' if 'MiniLM' in label else 'DistilBERT'
    backend = label.split()[-2] + ' ' + label.split()[-1]
    p50 = stats['p50']
    p99 = stats['p99']
    arm_est = f'~{p50*0.6:.1f}ms' if 'INT8' in label else 'N/A'
    cos = results.get(model.replace('-L6',''), {}).get('mean_fp32_int8', '-')
    cos_str = f'{cos:.5f}' if isinstance(cos, float) else '-'
    rows.append([model, backend, f'{p50:.2f}', f'{p99:.2f}', arm_est, cos_str])

headers = ['Model', 'Backend', 'p50 (ms)', 'p99 (ms)', 'ARM est.', 'cos(FP32,INT8)']
print(tabulate(rows, headers=headers, tablefmt='github'))

print('\nFile sizes:')
size_rows = [
    ['MiniLM-L6',  'FP32', f'{minilm_fp32_mb:.0f} MB'],
    ['MiniLM-L6',  'INT8', f'{minilm_int8_mb:.0f} MB'],
    ['DistilBERT', 'FP32', f'{dbert_fp32_mb:.0f} MB'],
    ['DistilBERT', 'INT8', f'{dbert_int8_mb:.0f} MB'],
]
print(tabulate(size_rows, headers=['Model', 'Format', 'Size'], tablefmt='github'))

## Section 15 — Save Results to JSON

Save all benchmark numbers to a JSON file so you can:
- Re-plot them without re-running the benchmark
- Import them into your paper's LaTeX tables
- Compare against results from the actual BF3 hardware later

## Section 16 — Generate `inputs/domains.bin` for C Inference

This section bridges the Python benchmark and the C binary (`infer_arm64`) running on the Bluefield 3.

**What it does (Part A — run now):**
1. Tokenizes the test domains with the same tokenizer used throughout the notebook
2. Writes `inputs/domains.bin` — the compact binary the C program reads
3. Runs Python ONNX INT8 inference (batch=1, L2-normalized) and saves the embeddings

**Binary layout written:**
```
Header:     [n_domains: int32][seq_len: int32][embed_dim: int32]
Per domain: [domain_len: int32][domain_str: char*][input_ids: int64*S][attn_mask: int64*S][tok_types: int64*S]
```

**What it does (Part B — run after the C binary):**
Cross-validates that `infer_arm64` on the BF3 produces the same embeddings as Python.

**Workflow:**
```
Part A  →  scp inputs/domains.bin bf3:/path/
           scp models/minilm/minilm_int8.onnx bf3:/path/
           # on BF3:
           mkdir -p outputs
           ./infer_arm64 minilm_int8.onnx domains.bin 200
           scp bf3:/path/outputs/c_embeddings.bin outputs/
Part B  →  run the cross-validation cell below
```

In [ ]:
import struct

INPUTS_DIR  = Path('inputs');   INPUTS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR = Path('outputs');  OUTPUTS_DIR.mkdir(exist_ok=True)

# ── tokenize with padding=max_length so every domain has the same seq_len ─────
# This is critical: the C binary reads a fixed [N × seq_len] layout.
print(f'Tokenizing {len(BENCH_DOMAINS)} domains (max_len={MAX_LEN}) ...')
enc = minilm_tok(
    BENCH_DOMAINS,
    padding='max_length',
    truncation=True,
    max_length=MAX_LEN,
    return_tensors='np',
)
input_ids  = enc['input_ids'].astype(np.int64)       # [N, MAX_LEN]
attn_masks = enc['attention_mask'].astype(np.int64)
tok_types  = enc.get('token_type_ids',
                     np.zeros_like(input_ids)).astype(np.int64)

N, S = input_ids.shape
EMBED_DIM = 384
print(f'  Shape: [{N} domains × {S} tokens]')

# ── write domains.bin ──────────────────────────────────────────────────────────
bin_path = INPUTS_DIR / 'domains.bin'
with open(bin_path, 'wb') as f:
    f.write(struct.pack('iii', N, S, EMBED_DIM))    # header
    for i, domain in enumerate(BENCH_DOMAINS):
        db = domain.encode('ascii')
        f.write(struct.pack('i', len(db)))           # domain string length
        f.write(db)                                  # domain string (no null)
        f.write(input_ids[i].tobytes())              # int64 × S
        f.write(attn_masks[i].tobytes())
        f.write(tok_types[i].tobytes())

print(f'  Written: {bin_path}  ({bin_path.stat().st_size:,} bytes)')

# ── run Python ONNX INT8 inference (batch=1, L2-normalised) ───────────────────
# This produces the ground-truth embeddings the C binary must match.
print(f'\nRunning Python ONNX INT8 (batch=1) on all {N} domains ...')

def _l2_norm(v: np.ndarray) -> np.ndarray:
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-9)

py_embeddings = []
for i in range(N):
    feeds = {
        'input_ids':      input_ids[i:i+1],
        'attention_mask': attn_masks[i:i+1],
        'token_type_ids': tok_types[i:i+1],
    }
    hidden = sess_minilm_int8.run(['last_hidden_state'], feeds)[0]  # [1, S, 384]
    emb    = mean_pool(hidden, attn_masks[i:i+1])                    # [1, 384]
    emb    = _l2_norm(emb)
    py_embeddings.append(emb[0])

py_embeddings = np.stack(py_embeddings)  # [N, 384]

# Save binary for cross-validation after C runs
py_bin = OUTPUTS_DIR / 'py_embeddings.bin'
with open(py_bin, 'wb') as f:
    f.write(struct.pack('ii', N, EMBED_DIM))
    f.write(py_embeddings.astype(np.float32).tobytes())
print(f'  Python embeddings saved → {py_bin}')

# Quick sanity: print first 4 dims for a few domains
print(f'\n  {"domain":<40}  {"tok_len":>7}  emb[0..3]')
print('  ' + '-'*72)
for i in [0, 5, -2, -1]:
    tlen = int((attn_masks[i] != 0).sum())
    print(f'  {BENCH_DOMAINS[i]:<40}  {tlen:>7}  {py_embeddings[i,:4]}')

print(f'\n✓ Part A done. Files to copy to Bluefield 3:')
print(f'   {bin_path}')
print(f'   {MINILM_ONNX_INT8}')
print(f'\nOn BF3:')
print(f'   mkdir -p outputs')
print(f'   ./infer_arm64 minilm_int8.onnx domains.bin 200')
print(f'   # then scp outputs/c_embeddings.bin back here and run Part B')

In [ ]:
c_bin  = OUTPUTS_DIR / 'c_embeddings.bin'
py_bin = OUTPUTS_DIR / 'py_embeddings.bin'

if not py_bin.exists():
    print(f'ERROR: {py_bin} not found — run Part A first')
elif not c_bin.exists():
    print(f'ERROR: {c_bin} not found — run infer_arm64 on BF3 and scp the file back')
else:
    with open(py_bin, 'rb') as f:
        N_py, D_py = struct.unpack('ii', f.read(8))
        py_emb = np.frombuffer(f.read(), dtype=np.float32).reshape(N_py, D_py)

    with open(c_bin, 'rb') as f:
        N_c, D_c = struct.unpack('ii', f.read(8))
        c_emb = np.frombuffer(f.read(), dtype=np.float32).reshape(N_c, D_c)

    print(f'Python: {N_py} domains × {D_py} dims')
    print(f'C:      {N_c}  domains × {D_c}  dims')
    if N_py != N_c or D_py != D_c:
        print('ERROR: shape mismatch — did you use the same domains.bin?')
    else:
        print(f'\n  {"domain":<45} {"cos(py,c)":>10} {"L2 diff":>10} {"✓"}')
        print('  ' + '-'*70)

        sims, l2s = [], []
        for i, d in enumerate(BENCH_DOMAINS[:N_c]):
            a, b = py_emb[i], c_emb[i]
            sim  = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
            l2   = float(np.linalg.norm(a - b))
            flag = '✓' if sim > 0.9999 else ('~' if sim > 0.999 else '✗')
            sims.append(sim); l2s.append(l2)
            print(f'  {d:<45} {sim:>10.7f} {l2:>10.6f}  {flag}')

        print('  ' + '-'*70)
        print(f'  {"MEAN":<45} {np.mean(sims):>10.7f} {np.mean(l2s):>10.6f}')
        good = sum(1 for s in sims if s > 0.9999)
        print(f'\n  {good}/{N_c} domains bit-equivalent (cos ≥ 0.9999) between C and Python ORT')